# 00 - Initial exploration of the data sources

This is the **first exploration**, kept as a record of how the project started.
It is not the reference: the code was then moved into `boxbox/` and the
results are explained in notebooks `01` to `04`.

What it shows, in order: a first request to TracingInsights, the first
versions of the functions of `boxbox/data.py` (`fetch_json`, `list_races`,
`find_race`, `list_drivers`, `load_driver_laps`, `load_laps`), a look at the
types and the missing values, a first version of the cleaning rules, and a
first pace table.

Internet is needed for the cells that download.

In [1]:
import pandas as pd
import requests

TRACING_URL = (
    "https://raw.githubusercontent.com/TracingInsights/2022/main/"
    "Bahrain%20Grand%20Prix/Race/LEC/laptimes.json"
)


def look(url, params=None):
    """Request a URL and print what comes back."""
    response = requests.get(url, params=params, timeout=20)
    print(response.status_code, response.url)
    if response.status_code != 200:
        return None
    return response.json()

## 1. TracingInsights: one driver, raw

In [2]:
# Test 1: TracingInsights
payload = look(TRACING_URL)
laps = pd.DataFrame(payload)
print(laps.shape)
laps.head(5)

200 https://raw.githubusercontent.com/TracingInsights/2022/main/Bahrain%20Grand%20Prix/Race/LEC/laptimes.json
(57, 42)


,time,lap,compound,stint,s1,s2,s3,ms1,ms2,ms3,...,ff1G,iacc,wT,wAT,wH,wP,wR,wTT,wWD,wWS
0,99.070,1,SOFT,1,None,42.120,23.984,None,None,None,...,False,False,3783.464,23.9,25.0,1010.2,False,29.1,20,0.5
1,97.853,2,SOFT,1,31.582,42.159,24.112,None,None,None,...,False,True,3903.470,23.8,26.0,1010.4,False,29.0,357,0.5
2,98.272,3,SOFT,1,31.635,42.404,24.233,None,None,None,...,False,True,3963.471,23.8,27.0,1010.4,False,29.0,12,0.5
3,98.414,4,SOFT,1,31.619,42.482,24.313,None,None,None,...,False,True,4083.471,23.8,29.0,1010.4,False,28.7,12,0.4
4,98.471,5,SOFT,1,31.59,42.565,24.316,None,None,None,...,False,True,4203.471,23.8,33.0,1010.4,False,28.5,316,0.4


## 2. First versions of the data functions

In [3]:
import os
import sys

# The notebooks live in notebooks/, the package in the project root.
sys.path.append(os.path.abspath(os.path.join("..")))

from boxbox.data import (
    DataLoadError,
    fetch_json,
    find_race,
    get_laps,
    get_session_folder,
    list_drivers,
    list_races,
    load_driver_laps,
    load_laps,
    prepare_laps,
    snapshot_path,
)

In [4]:
try:
    BASE = "https://raw.githubusercontent.com/TracingInsights/2022/main/"
    payload = fetch_json(BASE + "Monaco%20Grand%20Prix/Race/drivers.json")
    print(len(payload["drivers"]))
except DataLoadError as error:
    print(error)

20


In [5]:
try:
    races = list_races(2025)
    print(races)
except DataLoadError as error:
    print(error)

['Abu Dhabi Grand Prix', 'Australian Grand Prix', 'Austrian Grand Prix', 'Azerbaijan Grand Prix', 'Bahrain Grand Prix', 'Belgian Grand Prix', 'British Grand Prix', 'Canadian Grand Prix', 'Chinese Grand Prix', 'Dutch Grand Prix', 'Emilia Romagna Grand Prix', 'Hungarian Grand Prix', 'Italian Grand Prix', 'Japanese Grand Prix', 'Las Vegas Grand Prix', 'Mexico City Grand Prix', 'Miami Grand Prix', 'Monaco Grand Prix', 'Qatar Grand Prix', 'Saudi Arabian Grand Prix', 'Singapore Grand Prix', 'Spanish Grand Prix', 'São Paulo Grand Prix', 'United States Grand Prix']


In [6]:
race = find_race(2025, "MonACo")
print(race)

Monaco Grand Prix


In [7]:
list_drivers(2022, "Bahrain Grand Prix", "R")

['VER',
 'GAS',
 'PER',
 'ALO',
 'LEC',
 'STR',
 'MAG',
 'TSU',
 'ALB',
 'ZHO',
 'HUL',
 'RIC',
 'OCO',
 'NOR',
 'HAM',
 'MSC',
 'SAI',
 'LAT',
 'RUS',
 'BOT']

In [8]:
get_session_folder("Q")

'Qualifying'

In [9]:
laps_lec = load_driver_laps(2022, "Bahrain Grand Prix", "R", "LEC")
laps_lec.head(5)

,time,lap,compound,stint,s1,s2,s3,ms1,ms2,ms3,...,ff1G,iacc,wT,wAT,wH,wP,wR,wTT,wWD,wWS
0,99.070,1,SOFT,1,None,42.120,23.984,None,None,None,...,False,False,3783.464,23.9,25.0,1010.2,False,29.1,20,0.5
1,97.853,2,SOFT,1,31.582,42.159,24.112,None,None,None,...,False,True,3903.470,23.8,26.0,1010.4,False,29.0,357,0.5
2,98.272,3,SOFT,1,31.635,42.404,24.233,None,None,None,...,False,True,3963.471,23.8,27.0,1010.4,False,29.0,12,0.5
3,98.414,4,SOFT,1,31.619,42.482,24.313,None,None,None,...,False,True,4083.471,23.8,29.0,1010.4,False,28.7,12,0.4
4,98.471,5,SOFT,1,31.59,42.565,24.316,None,None,None,...,False,True,4203.471,23.8,33.0,1010.4,False,28.5,316,0.4


In [10]:
print(laps_lec.shape)

(57, 42)


## 3. The whole race, types and missing values

In [11]:
laps = load_laps(2022, "bahrain")
print(laps.columns.tolist())

['time', 'lap', 'compound', 'stint', 's1', 's2', 's3', 'ms1', 'ms2', 'ms3', 'life', 'pos', 'status', 'pb', 'sesT', 'drv', 'dNum', 'pout', 'pin', 's1T', 's2T', 's3T', 'vi1', 'vi2', 'vfl', 'vst', 'fresh', 'team', 'lST', 'lSD', 'del', 'delR', 'ff1G', 'iacc', 'wT', 'wAT', 'wH', 'wP', 'wR', 'wTT', 'wWD', 'wWS']


In [12]:
laps.dtypes

time         object
lap           int64
compound        str
stint         int64
s1           object
s2           object
s3           object
ms1             str
ms2             str
ms3             str
life          int64
pos          object
status          str
pb           object
sesT        float64
drv             str
dNum            str
pout         object
pin          object
s1T          object
s2T          object
s3T          object
vi1          object
vi2          object
vfl          object
vst          object
fresh          bool
team            str
lST         float64
lSD             str
del            bool
delR            str
ff1G           bool
iacc           bool
wT          float64
wAT         float64
wH          float64
wP          float64
wR             bool
wTT         float64
wWD           int64
wWS         float64
dtype: object

In [13]:
pd.set_option("display.max_columns", None)

laps.head(10)

,time,lap,compound,stint,s1,s2,s3,ms1,ms2,ms3,life,pos,status,pb,sesT,drv,dNum,pout,pin,s1T,s2T,s3T,vi1,vi2,vfl,vst,fresh,team,lST,lSD,del,delR,ff1G,iacc,wT,wAT,wH,wP,wR,wTT,wWD,wWS
0,100.236,1,SOFT,1,None,42.325,24.389,None,None,None,4,2,1,False,3855.340,VER,1,None,None,None,3831.046,3855.427,230.0,254.0,274.0,250.0,False,Red Bull Racing,3754.872,2022-03-20T15:03:34.889000000,False,None,False,False,3783.464,23.9,25.0,1010.2,False,29.1,20,0.5
1,97.88,2,SOFT,1,31.285,42.269,24.326,None,None,None,5,2,1,True,3953.220,VER,1,None,None,3886.662,3928.931,3953.257,230.0,252.0,276.0,296.0,False,Red Bull Racing,3855.340,2022-03-20T15:05:15.357000000,False,None,False,True,3903.470,23.8,26.0,1010.4,False,29.0,357,0.5
2,98.357,3,SOFT,1,31.499,42.474,24.384,None,None,None,6,2,1,False,4051.577,VER,1,None,None,3984.756,4027.23,4051.614,229.0,254.0,276.0,294.0,False,Red Bull Racing,3953.220,2022-03-20T15:06:53.237000000,False,None,False,True,3963.471,23.8,27.0,1010.4,False,29.0,12,0.5
3,98.566,4,SOFT,1,31.342,42.674,24.55,None,None,None,7,2,1,False,4150.143,VER,1,None,None,4082.956,4125.63,4150.18,231.0,250.0,276.0,297.0,False,Red Bull Racing,4051.577,2022-03-20T15:08:31.594000000,False,None,False,True,4083.471,23.8,29.0,1010.4,False,28.7,12,0.4
4,98.877,5,SOFT,1,31.498,42.854,24.525,None,None,None,8,2,1,False,4249.020,VER,1,None,None,4181.678,4224.532,4249.057,229.0,256.0,276.0,293.0,False,Red Bull Racing,4150.143,2022-03-20T15:10:10.160000000,False,None,False,True,4203.471,23.8,33.0,1010.4,False,28.5,316,0.4
5,98.94,6,SOFT,1,31.572,42.759,24.609,None,None,None,9,2,1,False,4347.960,VER,1,None,None,4280.629,4323.388,4347.997,229.0,255.0,275.0,288.0,False,Red Bull Racing,4249.020,2022-03-20T15:11:49.037000000,False,None,False,True,4263.469,23.7,33.0,1010.4,False,28.3,314,0.2
6,99.062,7,SOFT,1,31.637,42.845,24.58,None,None,None,10,2,1,False,4447.022,VER,1,None,None,4379.634,4422.479,4447.059,228.0,253.0,276.0,289.0,False,Red Bull Racing,4347.960,2022-03-20T15:13:27.977000000,False,None,False,True,4383.484,23.6,34.0,1010.4,False,28.3,37,0.1
7,99.092,8,SOFT,1,31.616,42.875,24.601,None,None,None,11,2,1,False,4546.114,VER,1,None,None,4478.675,4521.55,4546.151,229.0,253.0,276.0,290.0,False,Red Bull Racing,4447.022,2022-03-20T15:15:07.039000000,False,None,False,True,4503.469,23.5,35.0,1010.4,False,28.1,204,0.3
8,99.251,9,SOFT,1,31.625,42.957,24.669,None,None,None,12,2,1,False,4645.365,VER,1,None,None,4577.776,4620.733,4645.402,228.0,252.0,276.0,288.0,False,Red Bull Racing,4546.114,2022-03-20T15:16:46.131000000,False,None,False,True,4563.471,23.5,36.0,1010.4,False,28.1,339,0.1
9,99.392,10,SOFT,1,31.736,43.051,24.605,None,None,None,13,2,1,False,4744.757,VER,1,None,None,4677.138,4720.189,4744.794,228.0,248.0,275.0,288.0,False,Red Bull Racing,4645.365,2022-03-20T15:18:25.382000000,False,None,False,True,4683.471,23.3,36.0,1010.2,False,28.0,50,0.2


In [14]:
laps[["drv", "lap"]].iloc[50:60]

,drv,lap
50,VER,51
51,VER,52
52,VER,53
53,VER,54
54,GAS,1
55,GAS,2
56,GAS,3
57,GAS,4
58,GAS,5
59,GAS,6


In [15]:
laps.shape, laps["drv"].nunique()

((1125, 42), 20)

In [16]:
n_none = (laps["compound"] == "None").sum()
print(n_none)

0


In [17]:
raw = load_laps(2022, "bahrain")
laps = prepare_laps(raw)
laps.dtypes

time        float64
lap           int64
compound        str
stint         int64
life          int64
pos         float64
status          str
drv             str
pout        float64
pin         float64
fresh          bool
team            str
del            bool
iacc           bool
dtype: object

In [18]:
laps.isna().sum()

time           2
lap            0
compound       0
stint          0
life           0
pos            2
status         0
drv            0
pout        1067
pin         1066
fresh          0
team           0
del            0
iacc           0
dtype: int64

In [19]:
raw.shape

(1125, 42)

In [20]:
laps[laps["time"].isna() | laps["pos"].isna()]

,time,lap,compound,stint,life,pos,status,drv,pout,pin,fresh,team,del,iacc
98,NaN,45,HARD,3,13,NaN,164,GAS,NaN,NaN,True,AlphaTauri,False,False
155,NaN,57,SOFT,4,17,NaN,12,PER,NaN,NaN,False,Red Bull Racing,False,False


In [21]:
laps["compound"].value_counts()

compound
SOFT      675
MEDIUM    345
HARD      105
Name: count, dtype: int64

In [22]:
laps["status"].value_counts()

status
1      958
4       76
12      38
41      19
164     16
2        7
16       4
64       4
21       3
Name: count, dtype: int64

In [23]:
path = snapshot_path(2022, "Bahrain Grand Prix", "R")
print(path.name)

2022_bahrain_grand_prix_R.csv


## 4. First version of the cleaning rules

In [24]:
laps = prepare_laps(get_laps(2022, "bahrain"))

rules = {
    "pit lap": laps["pin"].notna() | laps["pout"].notna(),
    "lap 1": laps["lap"] == 1,
    "not green": laps["status"] != "1",
    "deleted": laps["del"],
    "inaccurate": ~laps["iacc"],
    "no time": laps["time"].isna(),
}
for name, bad in rules.items():
    print(name, bad.sum())

any_bad = pd.concat(rules, axis=1).any(axis=1)
print("clean laps:", (~any_bad).sum(), "of", len(laps))

pit lap 117
lap 1 20
not green 167
deleted 4
inaccurate 232
no time 2
clean laps: 854 of 1125


In [25]:
others = pd.concat(
    {name: bad for name, bad in rules.items() if name != "inaccurate"}, axis=1
).any(axis=1)

only_iacc = rules["inaccurate"] & ~others
print(only_iacc.sum())
laps[only_iacc].head(10)

0


,time,lap,compound,stint,life,pos,status,drv,pout,pin,fresh,team,del,iacc


In [26]:
from boxbox.cleaning import mark_clean_laps

laps = mark_clean_laps(prepare_laps(get_laps(2022, "bahrain")))
print(laps["is_clean"].sum(), "clean of", len(laps))  # expect 854 of 1125

clean = laps[laps["is_clean"]]
summary = clean.groupby("drv")["time"].agg(["count", "median", "mean", "min"])
summary.sort_values("median")

854 clean of 1125


,count,median,mean,min
drv,,,,
LEC,46,97.6930,97.599109,94.570
PER,42,98.0575,98.061405,96.089
VER,41,98.0910,97.847146,95.440
SAI,44,98.1505,98.017818,95.740
HAM,42,98.3805,98.303214,96.228
RUS,44,98.8215,98.626932,96.302
TSU,42,99.0400,99.354500,97.104
MAG,45,99.0580,99.092222,96.623
OCO,43,99.1880,99.289837,97.110


## 5. First pace table and status codes

In [27]:
from boxbox.analysis import race_pace

pace = race_pace(mark_clean_laps(prepare_laps(get_laps(2022, "bahrain"))))
pace

,drv,team,clean_laps,median_lap_time,gap
0,LEC,Ferrari,46,97.6930,0.0000
1,PER,Red Bull Racing,42,98.0575,0.3645
2,VER,Red Bull Racing,41,98.0910,0.3980
3,SAI,Ferrari,44,98.1505,0.4575
4,HAM,Mercedes,42,98.3805,0.6875
5,RUS,Mercedes,44,98.8215,1.1285
6,TSU,AlphaTauri,42,99.0400,1.3470
7,MAG,Haas F1 Team,45,99.0580,1.3650
8,OCO,Alpine,43,99.1880,1.4950
9,GAS,AlphaTauri,39,99.5130,1.8200


In [28]:
laps.groupby("status")["time"].median().sort_values()

status
12      99.3090
1       99.6525
2      100.3950
21     103.0970
16     104.8935
164    123.4630
4      126.2480
64     128.3655
41     135.0420
Name: time, dtype: float64